
# Trabajo Práctico Aprendizaje Automático 2

### Configuración Automática del Espacio de Trabajo (Estructura `src/`)

------------------------------>Correr esto únicamente si usamos la opción A que está en el README. <-------------------

In [1]:
import os

# Crear las carpetas de manera automática
os.makedirs('src/datos', exist_ok=True)
print("✅ Estructura 'src/datos' creada correctamente.")

✅ Estructura 'src/datos' creada correctamente.


Ahora creamos los archivos de inicialización vacíos necesarios para el módulo de Python.

In [2]:
# Crear archivos __init__.py vacíos
with open('src/__init__.py', 'w') as f:
    pass

with open('src/datos/__init__.py', 'w') as f:
    pass

print("✅ Archivos __init__.py creados con éxito.")

✅ Archivos __init__.py creados con éxito.


Por favor, pega el contenido correspondiente de tus scripts `carga.py`, `preprocesamiento.py`, y `analisis.py` en las celdas de abajo o utiliza el editor integrado de Colab haciendo doble clic sobre los archivos que ahora aparecen en el panel izquierdo dentro de `src/datos/` para guardar tu código de forma definitiva.

In [3]:
from google.colab import files
import os
import shutil

# Aseguramos que el directorio destino existe
destino = 'src/datos'


print("Selecciona los archivos para la carpeta src/datos/ (ej. carga.py, preprocesamiento.py, analisis.py):")
uploaded = files.upload()

for nombre_archivo, contenido in uploaded.items():
    # Definimos la ruta de destino completa
    ruta_destino = os.path.join(destino, nombre_archivo)

    # Movemos el archivo subido a la carpeta destino
    shutil.move(nombre_archivo, ruta_destino)
    print(f"✅ Archivo '{nombre_archivo}' guardado exitosamente en '{ruta_destino}'")

Selecciona los archivos para la carpeta src/datos/ (ej. carga.py, preprocesamiento.py, analisis.py):


Saving __init__.py to __init__.py
Saving analisis.py to analisis.py
Saving carga.py to carga.py
Saving preprocesamiento.py to preprocesamiento.py
✅ Archivo '__init__.py' guardado exitosamente en 'src/datos/__init__.py'
✅ Archivo 'analisis.py' guardado exitosamente en 'src/datos/analisis.py'
✅ Archivo 'carga.py' guardado exitosamente en 'src/datos/carga.py'
✅ Archivo 'preprocesamiento.py' guardado exitosamente en 'src/datos/preprocesamiento.py'


## Imports

In [4]:
%pip install datasets transformers torch pandas numpy scikit-learn matplotlib

In [6]:
# Actualizamos IPython para corregir la incompatibilidad de autoreload con Python 3.13 TUVE QUE AGREGAR ESTO PARA QUE ME CORRA EL BLOQUE DE ABAJO (Seba)
%pip install --upgrade --quiet ipython

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires ipython==7.34.0, but you have ipython 9.18.0 which is incompatible.


In [7]:
# Cargamos la extensión para recarga automática de módulos modificados
%load_ext autoreload
%autoreload 2

# Importamos las funciones de tus scripts subidos
from src.datos.carga import cargar_textos_wikipedia
from src.datos.preprocesamiento import construir_dataset, construir_embeddings_por_palabra, generar_matriz_features_rf
from src.datos.analisis import resumen_etiquetas

## Tokenizador

Utilizamos el código proporcionado.

In [8]:
from transformers import BertTokenizer, BertModel
import torch

model_name = "bert-base-multilingual-cased"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)
model.eval()

def get_multilingual_token_embedding(token: str):
    """
    Devuelve el embedding (estático) para el token.
    """
    token_id = tokenizer.convert_tokens_to_ids(token)

    if token_id is None or token_id == tokenizer.unk_token_id:
        print(f"❌El token '{token}' no pertenece al vocabulario de multilingual BERT.")
        return None

    embedding_vector = model.embeddings.word_embeddings.weight[token_id]

    #print(f"✅Token: '{token}' | ID: {token_id}")
    print(f"Embedding shape: {embedding_vector.shape}")
    return embedding_vector

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Dataset

Primero tenemos que conseguir textos para llevar a cabo la tarea. Vamos a utilizar un dataset de Wikipedia (por ahora) después lo podemos cambiar.

MODIFICACIÓN DE LA FUNCIÓN PARA CARGAR TEXTOS: agregué la opción de agregar preguntas en los textos que bajamos. Probé con 300 y funciona bien. Para el entrenamiento entiendo que tenemos que bajar más.

Traté de usar un dataset especial de preguntas o el de opensubtitles y me tiraba un error de HF... Entiendo que con esto está bien igual.

In [9]:
import re
# aleatorio=True toma artículos al azar en vez de los primeros 100 seguidos
textos_wiki, preguntas_wiki = cargar_textos_wikipedia(n_estandar=70, n_preguntas= 300, min_len=100, aleatorio=False)
muestra_textos = textos_wiki + preguntas_wiki

print(f"Se descargaron {len(muestra_textos)} textos de muestra.")

# Mostramos el primer ejemplo crudo
print("\nEjemplo de texto crudo descargado:")
print(muestra_textos[0][:250] + "...")

Iniciando descarga optimizada en streaming de Wikipedia en español (Versión: 20231101.es)...


README.md:   0%|          | 0.00/131k [00:00<?, ?B/s]

✅ Descarga finalizada: 70 textos estándar y 300 oraciones de preguntas obtenidas.
Se descargaron 370 textos de muestra.

Ejemplo de texto crudo descargado:
Lynx es un navegador web y cliente de gopher en modo texto.

Características 

Lynx es usado en terminales de cursor direccionable y celdas de caracteres, o emuladores de terminal (incluyendo terminales VT100 y paquetes de software para computadoras ...


Una vez que bajamos los parráfos de wikipedia, los vamos a procesar para armar el dataset que utilizaremos luego.

### Funciones de procesamiento

Están en `src/datos/preprocesamiento.py`:
- `normalizar_signos`: reemplaza o elimina los signos que no son etiquetas (`REEMPLAZOS`).
- `seccionar_palabra`: separa una palabra cruda en puntuación inicial, sub-palabras limpias y puntuación final.
- `determinar_capitalizacion`: clase de capitalización (0, 1, 2, 3) según la consigna.
- `procesar_parrafo_a_chunks`: tokeniza con BERT, asigna las etiquetas a cada sub-token y parte en chunks.
- `construir_dataset`: aplica lo anterior a todos los textos y concatena.

Reemplaza la extracción de puntuación que estaba dentro de `procesar_parrafo_a_chunks`. Antes solo se miraba el primer y el último carácter, así que en casos como `"¿Vienes?"` o `(Andorra),` se perdía la etiqueta, y los demás signos quedaban como tokens. Ahora se limpian todos los signos de los bordes, las palabras con guion se separan (`Castilla-La` → `Castilla La`) y se borran los signos internos (`EE.UU` → `EEUU`).

Aplicamos sobre la lista de parráfos bajados.

In [10]:
df_dataset_final = construir_dataset(muestra_textos, tokenizer, chunk_size=32)

print("Muestra del Dataset de Entrenamiento generado por Chunks:")
display(df_dataset_final.head(20))

Muestra del Dataset de Entrenamiento generado por Chunks:


,instancia_id,token_id,token,punt_inicial,punt_final,capitalización
0,1,66003,ly,,,1
1,1,10115,##n,,,1
2,1,10686,##x,,,1
3,1,10196,es,,,0
4,1,10119,un,,,0
5,1,20839,nave,,,0
6,1,43804,##gador,,,0
7,1,12998,web,,,0
8,1,193,y,,,0
9,1,69301,cliente,,,0


Veamos cuantas instacias tenemos

In [11]:
resumen_etiquetas(df_dataset_final)

Número total de tokens (filas en CSV): 213831
Número total de instancias (chunks): 6949

Punt. inicial:
 punt_inicial
(nada)    213481
¿            350
Name: count, dtype: int64 

Punt. final:
 punt_final
(nada)    196540
,          10639
.           6290
?            362
Name: count, dtype: int64 

Capitalización (por palabra):
 capitalización
0    133058
1     24085
2       227
3       891
Name: count, dtype: int64 

Instancias con '?': 4.63%
punt_inicial OK
punt_final OK
capitalización OK


In [12]:
# Verificar si algún chunk comienza con un sub-token
primeros_tokens = df_dataset_final.groupby("instancia_id").first()

chunks_rotos = primeros_tokens[
    primeros_tokens["token"].str.startswith("##")
]

print("Chunks que empiezan con ##:", len(chunks_rotos))

Chunks que empiezan con ##: 0


In [13]:
tamanios = df_dataset_final.groupby("instancia_id").size()

print("Tamaño mínimo:", tamanios.min())
print("Tamaño máximo:", tamanios.max())
print("Chunks con más de 32 tokens:", (tamanios > 32).sum())

Tamaño mínimo: 1
Tamaño máximo: 32
Chunks con más de 32 tokens: 0


## Preparación del Dataset para entrenamiento de Random Forest.

Utilizamos las funciones ```construir_embeddings_por_palabra```  y ```generar_matriz_features_rf``` para generar los datos de entrenamiento necesarios para entrenar el primer modelo del trabajo.

La idea es entrenar Randoms Forests, donde la entrada será el promedio de los embeddings de cada palabra. Con la primer función hacemos todos los cálculos. Con la segunda agrego la posibilidad de concatenar los embeddings de las palabras vecinas para darle más contexto al algoritmo.

In [15]:
# Esto lo tuve que poner para que no se sature la memoria del colab.
with torch.no_grad():
    df_palabras = construir_embeddings_por_palabra(
        df_dataset_final,
        get_multilingual_token_embedding
    )

# 2. Generar matriz X e y para entrenar el Random Forest
X, y_init, y_fin, y_cap = generar_matriz_features_rf(df_palabras, window_size=2)

print("=== VERIFICACIÓN DEL DATASET PROCESADO ===")
print("Número de palabras encontradas:", len(df_palabras))
print("Forma de la matriz de entradas X:", X.shape)
# Por ejemplo, si el embedding es de 768 y window_size=2, la dimensión será (N_palabras, 5 * 768) -> 3840 columnas

print("\nEjemplo del primer registro de palabra:")
print(df_palabras[['instancia_id', 'palabra', 'punt_inicial', 'punt_final', 'capitalización']].head(5))

Procesando embeddings de palabras:   0%|          | 0/158261 [00:00<?, ?it/s]

Se han truncado las últimas 5000 líneas del flujo de salida.
Embedding shape: torch.Size([768])
Embedding shape: torch.Size([768])
Embedding shape: torch.Size([768])
Embedding shape: torch.Size([768])
Embedding shape: torch.Size([768])
Embedding shape: torch.Size([768])
Embedding shape: torch.Size([768])
Embedding shape: torch.Size([768])
Embedding shape: torch.Size([768])
Embedding shape: torch.Size([768])
Embedding shape: torch.Size([768])
Embedding shape: torch.Size([768])
Embedding shape: torch.Size([768])
Embedding shape: torch.Size([768])
Embedding shape: torch.Size([768])
Embedding shape: torch.Size([768])
Embedding shape: torch.Size([768])
Embedding shape: torch.Size([768])
Embedding shape: torch.Size([768])
Embedding shape: torch.Size([768])
Embedding shape: torch.Size([768])
Embedding shape: torch.Size([768])
Embedding shape: torch.Size([768])
Embedding shape: torch.Size([768])
Embedding shape: torch.Size([768])
Embedding shape: torch.Size([768])
Embedding shape: torch.Size([